# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mkhlor006/Flyrank_internship_ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/mkhlor006/Flyrank_internship_ML.git

Cloning into 'Flyrank_internship_ML'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 152 (delta 58), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 1.89 MiB | 18.60 MiB/s, done.
Resolving deltas: 100% (58/58), done.


In [6]:
import os

print("Current directory:")
print(os.getcwd())

print("\nTop-level folders:")
print(os.listdir("."))

Current directory:
/content/Flyrank_internship_ML

Top-level folders:
['scripts', 'requirements.txt', 'data', '02_your_first_readable_model.ipynb', 'GUIDE.md', 'work', '.github', 'AGENTS.md', 'outputs', 'CLAUDE.md', 'README.md', 'LICENSE', '01_first_look_and_discovery.ipynb', 'submission', '.git', '.gitignore', 'notebooks', 'DATA_USE.md', 'docs', 'SETUP.md', 'skills']


In [7]:
!python scripts/01_prepare_features.py

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/Flyrank_internship_ML/data/processed/refresh_feature_vector.csv


In [8]:
import os

print(
    "Feature file exists:",
    os.path.exists("data/processed/refresh_feature_vector.csv")
)

Feature file exists: True


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



### Finding 1: Which Pages Will Grow?

The paper reports that a model trained on 96.6K pages that were clearly growing or declining achieved about 90% accuracy on unseen pages from the same brands and about 75% accuracy on brands the model had never seen before.

**Methodology question:** I would want to understand exactly how "growing" and "declining" were defined and over what future time window the labels were created. I would also check whether all features used by the model were available before that outcome period began. This would help confirm that the reported accuracy measures prediction of a future outcome rather than information that overlaps with the label.

I would also ask whether the unseen-brand evaluation uses a genuinely separated set of brands throughout model development, including feature selection and threshold choices. If so, the 75% result provides stronger evidence about performance on new brands; if not, the generalisation claim may be more optimistic than it appears.

### Finding 2: Refreshing Pages Actually Works

The paper reports that 7 of 9 strata showed statistically significant refresh lift, with the effect sizes reported separately by age and competition segment.

**Methodology question:** I would want to understand how the "refreshed" and "stale" groups were constructed and what outcome window was used after refresh. In particular, I would check how the label or outcome was defined and whether the comparison controls for differences between pages that were refreshed and pages that were not.

I would also check how the held-out evaluation was constructed and whether the same pages, brands, or information used to form the comparison groups could influence both the treatment definition and the outcome. This would help determine how far the result supports a directional refresh effect rather than only an observed association.

These are questions I would ask constructively because the answers determine how confidently the findings can be applied to new data and decisions.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Paper finding 1:")
print("Growth Prediction: 90% same-brand accuracy; 75% unseen-brand accuracy.")

print("\nPaper finding 2:")
print("Refreshing Pages Actually Works: 7 of 9 strata showed statistically significant lift.")

Paper finding 1:
Growth Prediction: 90% same-brand accuracy; 75% unseen-brand accuracy.

Paper finding 2:
Refreshing Pages Actually Works: 7 of 9 strata showed statistically significant lift.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*



In Week 5 I evaluated the model using a client-holdout split so that pages from the same client could not appear in both training and test sets. In this audit I compare that grouped result with a simple row-level split to show why the grouped design is more appropriate.

The row-level split is a useful diagnostic but is not my preferred estimate because pages from the same client can share characteristics. The client-grouped split better represents performance on clients the model did not see during training.

I keep Precision@50 as the main decision metric because my task is to prioritise the top pages for review rather than only classify every page equally.

In [10]:
import pandas as pd

feature_path = "data/processed/refresh_feature_vector.csv"

data = pd.read_csv(feature_path)

print("Prepared data shape:", data.shape)
print("Target counts:")
print(data["is_declining_label"].value_counts())
print("Clients:", data["client_id"].nunique())

Prepared data shape: (30000, 52)
Target counts:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Clients: 32


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42


def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))
    top_idx = np.argsort(scores)[::-1][:k]

    return float(y_true[top_idx].sum() / k)


# ---------------------------------------------------------
# Helper: build the same feature matrix used in Week 5
# ---------------------------------------------------------

NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier"
]


def make_features(frame):
    numeric = frame[NUMERIC_FEATURES].copy()

    numeric = (
        numeric
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    categorical = frame[CATEGORICAL_FEATURES].fillna("unknown").astype(str)

    categorical = pd.get_dummies(
        categorical,
        prefix=CATEGORICAL_FEATURES,
        dtype=float
    )

    return pd.concat(
        [
            numeric.reset_index(drop=True),
            categorical.reset_index(drop=True)
        ],
        axis=1
    )


# ---------------------------------------------------------
# 1. Row-level split
# ---------------------------------------------------------

row_train, row_test = train_test_split(
    data,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=data["is_declining_label"]
)

X_row_train = make_features(row_train)
X_row_test = make_features(row_test)

X_row_test = X_row_test.reindex(
    columns=X_row_train.columns,
    fill_value=0
)

y_row_train = row_train["is_declining_label"].astype(int)
y_row_test = row_test["is_declining_label"].astype(int)


row_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

row_model.fit(X_row_train, y_row_train)

row_scores = row_model.predict_proba(X_row_test)[:, 1]

row_precision50 = precision_at_k(
    y_row_test,
    row_scores,
    50
)


# ---------------------------------------------------------
# 2. Client-grouped split
# ---------------------------------------------------------

clients = data["client_id"].fillna("unknown").astype(str).unique()

rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(clients)

n_test_clients = max(
    1,
    int(round(len(shuffled_clients) * 0.20))
)

test_clients = set(
    shuffled_clients[:n_test_clients]
)

group_test_mask = (
    data["client_id"]
    .fillna("unknown")
    .astype(str)
    .isin(test_clients)
)

group_train = data.loc[~group_test_mask].copy()
group_test = data.loc[group_test_mask].copy()

X_group_train = make_features(group_train)
X_group_test = make_features(group_test)

X_group_test = X_group_test.reindex(
    columns=X_group_train.columns,
    fill_value=0
)

y_group_train = group_train["is_declining_label"].astype(int)
y_group_test = group_test["is_declining_label"].astype(int)


group_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

group_model.fit(X_group_train, y_group_train)

group_scores = group_model.predict_proba(X_group_test)[:, 1]

group_precision50 = precision_at_k(
    y_group_test,
    group_scores,
    50
)


# ---------------------------------------------------------
# Comparison
# ---------------------------------------------------------

comparison = pd.DataFrame({
    "Split": [
        "Row-level",
        "Client-grouped"
    ],
    "Precision@50": [
        row_precision50,
        group_precision50
    ]
})

print(comparison.round(3))

print("\nRow-level training rows:", len(row_train))
print("Row-level test rows:", len(row_test))

print("\nGrouped training clients:", group_train["client_id"].nunique())
print("Grouped test clients:", group_test["client_id"].nunique())

overlap = (
    set(group_train["client_id"].astype(str))
    & set(group_test["client_id"].astype(str))
)

print("Grouped client overlap:", len(overlap))

            Split  Precision@50
0       Row-level           0.9
1  Client-grouped           0.4

Row-level training rows: 24000
Row-level test rows: 6000

Grouped training clients: 26
Grouped test clients: 6
Grouped client overlap: 0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.